In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/ml-20m")

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
genome_tags = pd.read_csv(DATA_DIR / "genome-tags.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s", errors="coerce")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s", errors="coerce")

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)").astype(float)
movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies["genres_list"] = movies["genres"].fillna("(no genres listed)").str.split("|")

tags = tags.dropna(subset=["tag"]).copy()
tags["tag"] = tags["tag"].astype(str).str.strip()
tags = tags[tags["tag"] != ""].copy()

print("ratings shape:", ratings.shape)
print("movies shape:", movies.shape)
print("tags shape:", tags.shape)
print("genome_tags shape:", genome_tags.shape)
print("genome_scores shape:", genome_scores.shape)
print("links shape:", links.shape)

print("\nratings time range:", ratings["timestamp"].min(), "->", ratings["timestamp"].max())
print("tags time range   :", tags["timestamp"].min(), "->", tags["timestamp"].max())

print("\nnulls")
print("ratings:\n", ratings[["userId", "movieId", "rating", "timestamp"]].isna().sum())
print("movies:\n", movies[["movieId", "title", "genres", "year"]].isna().sum())
print("tags:\n", tags[["userId", "movieId", "tag", "timestamp"]].isna().sum())

ratings shape: (20000263, 4)
movies shape: (27278, 6)
tags shape: (465541, 4)
genome_tags shape: (1128, 2)
genome_scores shape: (11709768, 3)
links shape: (27278, 3)

ratings time range: 1995-01-09 11:46:44 -> 2015-03-31 06:40:02
tags time range   : 2005-12-24 13:00:10 -> 2015-03-31 03:09:12

nulls
ratings:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
movies:
 movieId     0
title       0
genres      0
year       22
dtype: int64
tags:
 userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


In [2]:

pair_counts = ratings.groupby(["userId", "movieId"]).size()

print("total rating rows:", len(ratings))
print("unique user-movie pairs:", len(pair_counts))
print("pairs with repeats:", (pair_counts > 1).sum())
print("rows inside repeated pairs:", pair_counts[pair_counts > 1].sum())
print("max repeats for one pair:", pair_counts.max())

ratings_last = (
    ratings
    .sort_values(["userId", "movieId", "timestamp"])
    .drop_duplicates(["userId", "movieId"], keep="last")
    .reset_index(drop=True)
)

print("\nafter keeping only the last rating per user-movie")
print("rows:", len(ratings_last))

for thr in [0.5, 3.5, 4.0]:
    x = ratings_last[ratings_last["rating"] >= thr].copy()
    user_len = x.groupby("userId").size()
    item_len = x.groupby("movieId").size()

    print(f"\nratings >= {thr}")
    print("rows:", len(x))
    print("users:", x["userId"].nunique())
    print("items:", x["movieId"].nunique())
    print("avg interactions per user:", round(user_len.mean(), 2))
    print("median interactions per user:", round(user_len.median(), 2))
    print("avg interactions per item:", round(item_len.mean(), 2))
    print("median interactions per item:", round(item_len.median(), 2))

total rating rows: 20000263
unique user-movie pairs: 20000263
pairs with repeats: 0
rows inside repeated pairs: 0
max repeats for one pair: 1

after keeping only the last rating per user-movie
rows: 20000263

ratings >= 0.5
rows: 20000263
users: 138493
items: 26744
avg interactions per user: 144.41
median interactions per user: 68.0
avg interactions per item: 747.84
median interactions per item: 18.0

ratings >= 3.5
rows: 12195566
users: 138362
items: 22884
avg interactions per user: 88.14
median interactions per user: 42.0
avg interactions per item: 532.93
median interactions per item: 16.0

ratings >= 4.0
rows: 9995410
users: 138287
items: 20720
avg interactions per user: 72.28
median interactions per user: 37.0
avg interactions per item: 482.4
median interactions per item: 15.0


In [3]:


def show_seq_stats(df, name):
    seq_len = (
        df.sort_values(["userId", "timestamp"])
          .groupby("userId")
          .size()
          .sort_values()
    )

    print(f"\n{name}")
    print("users:", len(seq_len))
    print(seq_len.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

    for L in [5, 10, 20, 50, 100, 200]:
        pct = (seq_len >= L).mean() * 100
        print(f"users with seq >= {L}: {pct:.2f}%")

for thr in [0.5, 3.5, 4.0]:
    x = ratings_last[ratings_last["rating"] >= thr].copy()
    show_seq_stats(x, f"sequence stats for ratings >= {thr}")


sequence stats for ratings >= 0.5
users: 138493
0.25      35.00
0.50      68.00
0.75     155.00
0.90     334.00
0.95     520.00
0.99    1113.08
dtype: float64
users with seq >= 5: 100.00%
users with seq >= 10: 100.00%
users with seq >= 20: 100.00%
users with seq >= 50: 61.60%
users with seq >= 100: 37.98%
users with seq >= 200: 19.37%

sequence stats for ratings >= 3.5
users: 138362
0.25     21.0
0.50     42.0
0.75    100.0
0.90    211.0
0.95    316.0
0.99    636.0
dtype: float64
users with seq >= 5: 99.36%
users with seq >= 10: 95.95%
users with seq >= 20: 77.33%
users with seq >= 50: 44.75%
users with seq >= 100: 25.04%
users with seq >= 200: 10.81%

sequence stats for ratings >= 4.0
users: 138287
0.25     19.0
0.50     37.0
0.75     84.0
0.90    170.0
0.95    249.0
0.99    492.0
dtype: float64
users with seq >= 5: 98.84%
users with seq >= 10: 93.86%
users with seq >= 20: 73.58%
users with seq >= 50: 40.50%
users with seq >= 100: 20.71%
users with seq >= 200: 7.62%


In [4]:


tag_stats = (
    tags.groupby("movieId")
        .agg(
            raw_tag_rows=("tag", "size"),
            raw_unique_tags=("tag", "nunique")
        )
        .reset_index()
)

genome_stats = (
    genome_scores.groupby("movieId")
                 .agg(
                     genome_dims=("tagId", "size"),
                     genome_mean_rel=("relevance", "mean")
                 )
                 .reset_index()
)

movie_sem = (
    movies[["movieId", "clean_title", "genres", "year"]]
    .merge(tag_stats, on="movieId", how="left")
    .merge(genome_stats, on="movieId", how="left")
)

movie_sem["has_raw_tags"] = movie_sem["raw_tag_rows"].fillna(0) > 0
movie_sem["has_genome"] = movie_sem["genome_dims"].fillna(0) > 0
movie_sem["has_both"] = movie_sem["has_raw_tags"] & movie_sem["has_genome"]

print("all movies")
print("total movies:", len(movie_sem))
print("movies with raw tags:", int(movie_sem["has_raw_tags"].sum()))
print("movies with genome:", int(movie_sem["has_genome"].sum()))
print("movies with both:", int(movie_sem["has_both"].sum()))
print("mean raw tag rows per movie:", round(movie_sem["raw_tag_rows"].fillna(0).mean(), 2))
print("mean unique raw tags per movie:", round(movie_sem["raw_unique_tags"].fillna(0).mean(), 2))

covered_genome = movie_sem[movie_sem["has_genome"]]
if len(covered_genome) > 0:
    print("genome dims min/max:", int(covered_genome["genome_dims"].min()), int(covered_genome["genome_dims"].max()))

active_movie_ids = ratings_last["movieId"].unique()
active_sem = movie_sem[movie_sem["movieId"].isin(active_movie_ids)].copy()

print("\nactive movies only")
print("active movies:", len(active_sem))
print("active movies with raw tags:", int(active_sem["has_raw_tags"].sum()))
print("active movies with genome:", int(active_sem["has_genome"].sum()))
print("active movies with both:", int(active_sem["has_both"].sum()))

print("\nmost common raw tags")
print(tags["tag"].str.lower().value_counts().head(30))

all movies
total movies: 27278
movies with raw tags: 19545
movies with genome: 10381
movies with both: 9827
mean raw tag rows per movie: 17.07
mean unique raw tags per movie: 7.35
genome dims min/max: 1128 1128

active movies only
active movies: 26744
active movies with raw tags: 19011
active movies with genome: 10370
active movies with both: 9816

most common raw tags
tag
sci-fi                3576
based on a book       3307
atmospheric           3169
comedy                3078
action                3068
nudity (topless)      2646
surreal               2528
twist ending          2367
bd-r                  2334
funny                 2253
quirky                2034
dystopia              2015
classic               1971
stylized              1944
dark comedy           1910
romance               1874
fantasy               1850
psychology            1763
time travel           1572
disturbing            1524
visually appealing    1511
aliens                1439
social commentary     1424
tho

In [5]:


def split_preview(df, name, min_hist=5):
    df = df.sort_values(["userId", "timestamp"]).copy()

    user_counts = df.groupby("userId").size()
    keep_users = user_counts[user_counts >= min_hist].index
    df = df[df["userId"].isin(keep_users)].copy()

    df["pos"] = df.groupby("userId").cumcount()
    df["n"] = df.groupby("userId")["movieId"].transform("size")

    train = df[df["pos"] < df["n"] - 2].copy()
    val = df[df["pos"] == df["n"] - 2].copy()
    test = df[df["pos"] == df["n"] - 1].copy()

    train_items = set(train["movieId"].unique())
    val_unseen = (~val["movieId"].isin(train_items)).mean() * 100
    test_unseen = (~test["movieId"].isin(train_items)).mean() * 100

    print(f"\n{name}")
    print("users kept:", df["userId"].nunique())
    print("train rows:", len(train))
    print("val rows:", len(val))
    print("test rows:", len(test))
    print("train items:", train["movieId"].nunique())
    print("val unseen item %:", round(val_unseen, 2))
    print("test unseen item %:", round(test_unseen, 2))

for thr in [3.5, 4.0]:
    x = ratings_last[ratings_last["rating"] >= thr].copy()
    split_preview(x, f"chrono split preview for ratings >= {thr}", min_hist=5)


chrono split preview for ratings >= 3.5
users kept: 137477
train rows: 11917990
val rows: 137477
test rows: 137477
train items: 22768
val unseen item %: 0.04
test unseen item %: 0.04

chrono split preview for ratings >= 4.0
users kept: 136677
train rows: 9717328
val rows: 136677
test rows: 136677
train items: 20575
val unseen item %: 0.04
test unseen item %: 0.07


In [6]:

#final data check for preprocessing
x = ratings[ratings["rating"] >= 3.5].copy()

print("rows:", len(x))
print("users:", x["userId"].nunique())
print("items:", x["movieId"].nunique())

user_counts = x.groupby("userId").size()
item_counts = x.groupby("movieId").size()

print("\nuser count quantiles")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nitem count quantiles")
print(item_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nusers with at least 5 interactions:", (user_counts >= 5).sum())
print("items with at least 5 interactions:", (item_counts >= 5).sum())
print("items with at least 10 interactions:", (item_counts >= 10).sum())

rows: 12195566
users: 138362
items: 22884

user count quantiles
0.25     21.0
0.50     42.0
0.75    100.0
0.90    211.0
0.95    316.0
0.99    636.0
dtype: float64

item count quantiles
0.25        3.00
0.50       16.00
0.75      142.00
0.90      869.00
0.95     2476.85
0.99    10569.70
dtype: float64

users with at least 5 interactions: 137477
items with at least 5 interactions: 15405
items with at least 10 interactions: 12988


In [7]:


#top tags per movie preview
tag_counts = (
    tags.assign(tag_lower=tags["tag"].str.lower())
        .groupby(["movieId", "tag_lower"])
        .size()
        .reset_index(name="count")
        .sort_values(["movieId", "count", "tag_lower"], ascending=[True, False, True])
)

top_tags_preview = tag_counts.groupby("movieId").head(5).copy()

sample_movies = [1, 50, 260, 296, 318]
print(
    top_tags_preview[top_tags_preview["movieId"].isin(sample_movies)]
    .sort_values(["movieId", "count"], ascending=[True, False])
    .to_string(index=False)
)

 movieId           tag_lower  count
       1               pixar     79
       1           animation     49
       1              disney     28
       1           tom hanks     23
       1  computer animation     20
      50        twist ending    139
      50        kevin spacey     89
      50            suspense     58
      50         complicated     57
      50     organized crime     44
     260              sci-fi     86
     260               space     64
     260       harrison ford     49
     260             fantasy     43
     260           adventure     36
     296   quentin tarantino    217
     296         dark comedy    117
     296           nonlinear    113
     296   samuel l. jackson    102
     296 multiple storylines     93
     318      morgan freeman    110
     318              prison    104
     318        twist ending     90
     318        stephen king     85
     318   thought-provoking     75


In [8]:

#year bucket preview
movies["year_bucket"] = pd.cut(
    movies["year"],
    bins=[1900, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020],
    include_lowest=True
)

print(movies["year_bucket"].value_counts(dropna=False).sort_index())


year_bucket
(1899.999, 1950.0]    2542
(1950.0, 1960.0]      1395
(1960.0, 1970.0]      1717
(1970.0, 1980.0]      2059
(1980.0, 1990.0]      2725
(1990.0, 2000.0]      4671
(2000.0, 2010.0]      8224
(2010.0, 2020.0]      3909
NaN                     36
Name: count, dtype: int64


In [9]:

# filtered semantic coverage
filtered_movie_ids = set(x["movieId"].unique())

movie_sem_small = movies[["movieId", "clean_title", "genres", "year", "year_bucket"]].copy()
movie_sem_small["has_raw_tags"] = movie_sem_small["movieId"].isin(tags["movieId"].unique())
movie_sem_small["has_genome"] = movie_sem_small["movieId"].isin(genome_scores["movieId"].unique())

movie_sem_small = movie_sem_small[movie_sem_small["movieId"].isin(filtered_movie_ids)].copy()
movie_sem_small["has_both"] = movie_sem_small["has_raw_tags"] & movie_sem_small["has_genome"]

print("filtered items:", len(movie_sem_small))
print("with raw tags:", int(movie_sem_small["has_raw_tags"].sum()))
print("with genome:", int(movie_sem_small["has_genome"].sum()))
print("with both:", int(movie_sem_small["has_both"].sum()))

filtered items: 22884
with raw tags: 17791
with genome: 10362
with both: 9810


____
# 3

In [10]:

# make the final filtered interaction table
x = ratings[ratings["rating"] >= 3.5].copy()

def kcore_filter(df, user_min=5, item_min=5):
    df = df.copy()

    while True:
        n_before = len(df)

        good_users = df["userId"].value_counts()
        good_users = good_users[good_users >= user_min].index
        df = df[df["userId"].isin(good_users)].copy()

        good_items = df["movieId"].value_counts()
        good_items = good_items[good_items >= item_min].index
        df = df[df["movieId"].isin(good_items)].copy()

        if len(df) == n_before:
            break

    return df

interactions = kcore_filter(x, user_min=5, item_min=5)
interactions = interactions.sort_values(["userId", "timestamp"]).reset_index(drop=True)

print("final filtered interactions")
print("rows:", len(interactions))
print("users:", interactions["userId"].nunique())
print("items:", interactions["movieId"].nunique())

user_counts = interactions.groupby("userId").size()
item_counts = interactions.groupby("movieId").size()

print("\nuser count quantiles")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nitem count quantiles")
print(item_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

final filtered interactions
rows: 12178563
users: 137474
items: 15405

user count quantiles
0.25     21.0
0.50     43.0
0.75    100.0
0.90    211.0
0.95    317.0
0.99    636.0
dtype: float64

item count quantiles
0.25       15.00
0.50       64.00
0.75      332.00
0.90     1620.60
0.95     3903.60
0.99    13399.08
dtype: float64


In [11]:

# map userId and movieId into model ids
user_ids = np.sort(interactions["userId"].unique())
movie_ids = np.sort(interactions["movieId"].unique())

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {m: i + 1 for i, m in enumerate(movie_ids)}   # 0 is kept for padding
idx2item = {i: m for m, i in item2idx.items()}

interactions["user_idx"] = interactions["userId"].map(user2idx)
interactions["item_idx"] = interactions["movieId"].map(item2idx)

print("num users:", len(user2idx))
print("num items:", len(item2idx))
print("\ninteractions preview")
print(
    interactions[["userId", "movieId", "user_idx", "item_idx", "rating", "timestamp"]]
    .head(10)
    .to_string(index=False)
)

num users: 137474
num items: 15405

interactions preview
 userId  movieId  user_idx  item_idx  rating           timestamp
      1      924         0       885     3.5 2004-09-10 03:06:38
      1      919         0       880     3.5 2004-09-10 03:07:01
      1     2683         0      2530     3.5 2004-09-10 03:07:30
      1     1584         0      1500     3.5 2004-09-10 03:07:36
      1     1079         0      1034     4.0 2004-09-10 03:07:45
      1     2959         0      2801     4.0 2004-09-10 03:08:18
      1      337         0       331     3.5 2004-09-10 03:08:29
      1     3996         0      3780     4.0 2004-09-10 03:08:47
      1      151         0       147     4.0 2004-09-10 03:08:54
      1      112         0       110     3.5 2004-09-10 03:09:00


In [12]:

#build train / val / test sequences for SASRec
user_seq = (
    interactions.sort_values(["user_idx", "timestamp"])
    .groupby("user_idx")["item_idx"]
    .apply(list)
)

split_rows = []

for user_idx, seq in user_seq.items():
    split_rows.append({
        "user_idx": user_idx,
        "train_seq": seq[:-2],
        "val_seq": seq[:-2],
        "val_target": seq[-2],
        "test_seq": seq[:-1],
        "test_target": seq[-1],
        "full_len": len(seq)
    })

splits = pd.DataFrame(split_rows)

print("number of users in splits:", len(splits))
print("train seq length quantiles")
print(splits["train_seq"].apply(len).quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nsplit preview")
print(splits.head(5).to_string(index=False))


number of users in splits: 137474
train seq length quantiles
0.25     19.0
0.50     41.0
0.75     98.0
0.90    209.0
0.95    315.0
0.99    634.0
Name: train_seq, dtype: float64

split preview
 user_idx                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                train_seq                                                                                                                                                                                                                      

In [13]:


# build item metadata table
year_labels = [
    "<=1950",
    "1951-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-2000",
    "2001-2010",
    "2011-2020"
]

movies["year_bucket"] = pd.cut(
    movies["year"],
    bins=[0, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020],
    labels=year_labels,
    include_lowest=True
)

movies["year_bucket"] = movies["year_bucket"].astype("object")
movies["year_bucket"] = movies["year_bucket"].fillna("unknown")

tag_counts = (
    tags.assign(tag_lower=tags["tag"].str.lower())
    .groupby(["movieId", "tag_lower"])
    .size()
    .reset_index(name="count")
    .sort_values(["movieId", "count", "tag_lower"], ascending=[True, False, True])
)

top_tags = tag_counts.groupby("movieId").head(5).copy()

top_tags_text = (
    top_tags.groupby("movieId")["tag_lower"]
    .apply(lambda s: " | ".join(s.tolist()))
    .reset_index(name="top_tags_text")
)

item_meta = movies[movies["movieId"].isin(movie_ids)].copy()
item_meta = item_meta.merge(top_tags_text, on="movieId", how="left")
item_meta["top_tags_text"] = item_meta["top_tags_text"].fillna("")

item_meta["genres_text"] = item_meta["genres"].fillna("").str.replace("|", " ", regex=False)

item_meta["movie_text"] = (
    item_meta["clean_title"].fillna("") + ". " +
    item_meta["genres_text"].fillna("") + ". " +
    item_meta["top_tags_text"].fillna("")
).str.strip()

genre_dummies = item_meta["genres"].str.get_dummies(sep="|")
item_meta = pd.concat([item_meta, genre_dummies], axis=1)

item_meta["item_idx"] = item_meta["movieId"].map(item2idx)
item_meta = item_meta.sort_values("item_idx").reset_index(drop=True)

print("item_meta shape:", item_meta.shape)
print("\nitem_meta preview")
print(
    item_meta[["item_idx", "movieId", "clean_title", "year_bucket", "top_tags_text", "movie_text"]]
    .head(10)
    .to_string(index=False)
)

item_meta shape: (15405, 31)

item_meta preview
 item_idx  movieId                 clean_title year_bucket                                                                         top_tags_text                                                                                                              movie_text
        1        1                   Toy Story   1991-2000                           pixar | animation | disney | tom hanks | computer animation     Toy Story. Adventure Animation Children Comedy Fantasy. pixar | animation | disney | tom hanks | computer animation
        2        2                     Jumanji   1991-2000                         robin williams | fantasy | time travel | animals | board game                      Jumanji. Adventure Children Fantasy. robin williams | fantasy | time travel | animals | board game
        3        3            Grumpier Old Men   1991-2000                        moldy | old | sequel | clv | comedinha de velhinhos engraã§ada             

In [14]:


# reduce genome features to a smaller vector
from sklearn.decomposition import TruncatedSVD

genome_small = genome_scores[genome_scores["movieId"].isin(movie_ids)].copy()

genome_wide = (
    genome_small.pivot(index="movieId", columns="tagId", values="relevance")
    .fillna(0)
    .astype("float32")
)

svd_dim = 64

svd = TruncatedSVD(n_components=svd_dim, random_state=42)
genome_reduced = svd.fit_transform(genome_wide)

genome_reduced = pd.DataFrame(
    genome_reduced,
    columns=[f"g{i}" for i in range(svd_dim)]
)
genome_reduced["movieId"] = genome_wide.index.values
genome_reduced["item_idx"] = genome_reduced["movieId"].map(item2idx)

genome_full = pd.DataFrame({"movieId": movie_ids})
genome_full["item_idx"] = genome_full["movieId"].map(item2idx)
genome_full = genome_full.merge(genome_reduced, on=["movieId", "item_idx"], how="left")
genome_full = genome_full.fillna(0).sort_values("item_idx").reset_index(drop=True)

print("movies with raw genome:", genome_wide.shape[0])
print("genome_full shape:", genome_full.shape)
print("svd explained variance sum:", round(svd.explained_variance_ratio_.sum(), 4))

print("\ngenome preview")
print(genome_full.head(5).to_string(index=False))

movies with raw genome: 10302
genome_full shape: (15405, 66)
svd explained variance sum: 0.7323

genome preview
 movieId  item_idx       g0       g1        g2        g3        g4        g5        g6       g7        g8        g9       g10       g11       g12       g13       g14       g15       g16       g17       g18       g19      g20       g21       g22       g23       g24       g25       g26       g27      g28       g29       g30       g31       g32       g33       g34       g35       g36       g37       g38       g39       g40       g41      g42       g43       g44       g45       g46       g47       g48       g49       g50       g51       g52       g53       g54       g55       g56       g57       g58      g59       g60       g61       g62       g63
       1         1 6.717240 1.186661 -2.051676  0.738828  0.900787  1.761749  1.217470 0.611673 -1.685390  0.833009  1.552775 -0.428605 -0.491475  0.214898 -1.183673 -0.719197  0.060525 -0.363902  0.404684  0.052617 0.250260 -0.693751 -

In [15]:


# save the processed objects
from pathlib import Path

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

interactions.to_pickle(OUT_DIR / "interactions_35_kcore5.pkl")
splits.to_pickle(OUT_DIR / "splits_35_kcore5.pkl")
item_meta.to_pickle(OUT_DIR / "item_meta_35_kcore5.pkl")
genome_full.to_pickle(OUT_DIR / "genome_svd64_35_kcore5.pkl")

pd.DataFrame({
    "userId": list(user2idx.keys()),
    "user_idx": list(user2idx.values())
}).to_pickle(OUT_DIR / "user_map.pkl")

pd.DataFrame({
    "movieId": list(item2idx.keys()),
    "item_idx": list(item2idx.values())
}).to_pickle(OUT_DIR / "item_map.pkl")

print("saved files in:", OUT_DIR)
print(sorted([p.name for p in OUT_DIR.iterdir()]))

saved files in: ../data/processed
['genome_svd64_35_kcore5.pkl', 'interactions_35_kcore5.pkl', 'item_map.pkl', 'item_meta_35_kcore5.pkl', 'splits_35_kcore5.pkl', 'user_map.pkl']


_______
# Model

In [16]:

# imports, config, load processed data
import math
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PROC_DIR = Path("../data/processed")

interactions = pd.read_pickle(PROC_DIR / "interactions_35_kcore5.pkl")
splits = pd.read_pickle(PROC_DIR / "splits_35_kcore5.pkl")
item_meta = pd.read_pickle(PROC_DIR / "item_meta_35_kcore5.pkl")
genome_full = pd.read_pickle(PROC_DIR / "genome_svd64_35_kcore5.pkl")
item_map = pd.read_pickle(PROC_DIR / "item_map.pkl")
user_map = pd.read_pickle(PROC_DIR / "user_map.pkl")

num_users = len(user_map)
num_items = len(item_map)

MAX_LEN = 200
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 10

print("num_users:", num_users)
print("num_items:", num_items)
print("splits:", splits.shape)



device: cuda
num_users: 137474
num_items: 15405
splits: (137474, 7)


In [23]:

# small helpers and datasets
def pad_seq(seq, max_len=200):
    seq = seq[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    out[-len(seq):] = seq
    return out

class TrainDataset(Dataset):
    def __init__(self, splits_df, max_len=200):
        self.max_len = max_len
        self.seqs = splits_df["train_seq"].tolist()

        keep = []
        for seq in self.seqs:
            if len(seq) >= 3:
                keep.append(seq)

        self.data = keep

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]

        tokens = seq[:-1]
        targets = seq[1:]

        tokens = pad_seq(tokens, self.max_len)
        targets = pad_seq(targets, self.max_len)

        return {
            "seq": torch.tensor(tokens, dtype=torch.long),
            "target_seq": torch.tensor(targets, dtype=torch.long),
        }

class EvalDataset(Dataset):
    def __init__(self, splits_df, mode="val", max_len=200):
        self.max_len = max_len
        self.mode = mode

        if mode == "val":
            self.seq_col = "val_seq"
            self.target_col = "val_target"
        else:
            self.seq_col = "test_seq"
            self.target_col = "test_target"

        self.seq = splits_df[self.seq_col].tolist()
        self.target = splits_df[self.target_col].tolist()

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return {
            "seq": torch.tensor(pad_seq(self.seq[idx], self.max_len), dtype=torch.long),
            "target": torch.tensor(self.target[idx], dtype=torch.long),
        }

train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 137474
val users: 137474
test users: 137474


In [27]:


#SASRec base model
class SASRecBase(nn.Module):
    def __init__(self, num_items, max_len=200, d_model=128, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        mask = torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)
        return mask

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)
        x = self.item_emb(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

model = SASRecBase(
    num_items=num_items,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print(model)

SASRecBase(
  (item_emb): Embedding(15406, 128, padding_idx=0)
  (pos_emb): Embedding(201, 128, padding_idx=0)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (emb_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)

In [28]:

# train and eval helpers
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target_seq = batch["target_seq"].to(device)

        optimizer.zero_grad()

        logits = model.predict_all_positions(seq)   # [B, L, num_items+1]

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_seq.reshape(-1),
            ignore_index=0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target = batch["target"].to(device)

        logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }

In [29]:

# train the base SASRec model
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_val_ndcg = -1
best_state = None

train_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate_topk(model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg:
        best_val_ndcg = val_metrics["NDCG@10"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

total_train_sec = time.perf_counter() - train_start
print("\ntraining finished in", round(total_train_sec, 2), "seconds")

epoch 01 | loss 6.7041 | HR@10 0.0209 | NDCG@10 0.0098 | MRR@10 0.0064 | 191.6s
epoch 02 | loss 6.1218 | HR@10 0.0232 | NDCG@10 0.0110 | MRR@10 0.0073 | 191.8s
epoch 03 | loss 6.0189 | HR@10 0.0238 | NDCG@10 0.0115 | MRR@10 0.0078 | 192.8s
epoch 04 | loss 5.9657 | HR@10 0.0246 | NDCG@10 0.0119 | MRR@10 0.0081 | 191.8s
epoch 05 | loss 5.9331 | HR@10 0.0247 | NDCG@10 0.0120 | MRR@10 0.0081 | 191.9s
epoch 06 | loss 5.9106 | HR@10 0.0253 | NDCG@10 0.0123 | MRR@10 0.0084 | 191.7s
epoch 07 | loss 5.8942 | HR@10 0.0255 | NDCG@10 0.0125 | MRR@10 0.0086 | 191.4s
epoch 08 | loss 5.8816 | HR@10 0.0256 | NDCG@10 0.0124 | MRR@10 0.0085 | 191.9s
epoch 09 | loss 5.8716 | HR@10 0.0255 | NDCG@10 0.0124 | MRR@10 0.0084 | 191.9s
epoch 10 | loss 5.8635 | HR@10 0.0259 | NDCG@10 0.0126 | MRR@10 0.0086 | 191.7s

training finished in 1918.49 seconds


In [30]:

# load best epoch and evaluate on test
model.load_state_dict(best_state)

test_start = time.perf_counter()
test_metrics = evaluate_topk(model, test_loader, device, k=10)
test_sec = time.perf_counter() - test_start

print("best validation NDCG@10:", round(best_val_ndcg, 6))
print("test metrics:")
for k, v in test_metrics.items():
    print(k, ":", round(v, 6))
print("test eval seconds:", round(test_sec, 2))

history_df = pd.DataFrame(history)
history_df

best validation NDCG@10: 0.012623
test metrics:
HR@10 : 0.022884
NDCG@10 : 0.011223
MRR@10 : 0.007718
Recall@10 : 0.022884
test eval seconds: 48.77


,epoch,train_loss,epoch_sec,HR@10,NDCG@10,MRR@10,Recall@10
0,1,6.704104,191.577443,0.020877,0.009772,0.006448,0.020877
1,2,6.121785,191.775480,0.023161,0.010992,0.007348,0.023161
2,3,6.018896,192.835710,0.023830,0.011518,0.007827,0.023830
3,4,5.965745,191.760316,0.024557,0.011869,0.008066,0.024557
4,5,5.933090,191.864506,0.024732,0.011970,0.008143,0.024732
5,6,5.910595,191.739416,0.025336,0.012275,0.008353,0.025336
6,7,5.894190,191.420478,0.025547,0.012518,0.008597,0.025547
7,8,5.881551,191.854300,0.025561,0.012407,0.008459,0.025561
8,9,5.871609,191.864351,0.025503,0.012372,0.008427,0.025503
9,10,5.863467,191.733373,0.025889,0.012623,0.008637,0.025889


In [31]:


# save model and results
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = Path("../reports/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(best_state, MODEL_DIR / "sasrec_base_id_only.pt")
history_df.to_csv(RESULT_DIR / "sasrec_base_history.csv", index=False)

pd.DataFrame([{
    "model": "sasrec_base_id_only",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS,
    "train_total_sec": total_train_sec,
    "test_eval_sec": test_sec,
    **test_metrics
}]).to_csv(RESULT_DIR / "sasrec_base_test_metrics.csv", index=False)

print("saved model:", MODEL_DIR / "sasrec_base_id_only.pt")
print("saved history:", RESULT_DIR / "sasrec_base_history.csv")
print("saved test metrics:", RESULT_DIR / "sasrec_base_test_metrics.csv")

saved model: ../models/sasrec_base_id_only.pt
saved history: ../reports/results/sasrec_base_history.csv
saved test metrics: ../reports/results/sasrec_base_test_metrics.csv
